In [ ]:
# !pip install pandas tqdm

In [ ]:
# 1bc6d1da-5d01-4bd7-8e95-805197693ab0

In [9]:
import requests
import json

API_KEY = "1bc6d1da-5d01-4bd7-8e95-805197693ab0"
BASE_URL = "https://www.youthcenter.go.kr/go/ythip/getPlcy"

params = {
    'apiKeyNm': API_KEY,
    'pageNum': 1,
    'pageSize': 5,
    'rtnType': 'json'
}

response = requests.get(BASE_URL, params=params)

# 응답 내용 그대로 출력
print("⚠️ 응답 원문:")
print(response.text)

⚠️ 응답 원문:
{"resultCode":200,"resultMessage":"성공적으로 데이터를 가지고 왔습니다.","result":{"pagging":{"totCount":3678,"pageNum":1,"pageSize":5},"youthPolicyList":[{"plcyNo":"20250707005400211155","bscPlanCycl":"1","bscPlanPlcyWayNo":"001","bscPlanFcsAsmtNo":"001","bscPlanAsmtNo":"002","pvsnInstGroupCd":"0054002","plcyPvsnMthdCd":"0042002","plcyAprvSttsCd":"0044002","plcyNm":"공공기관 맞춤형 취업지원사업","plcyKywdNm":"교육지원","plcyExplnCn":"울산경남 지역 공공기관의 울산 청년 기업기회 확대를 위해 공공기관 취업을 준비하는 청년들의 맞춤형 교육 지원","lclsfNm":"일자리","mclsfNm":"취업","plcySprtCn":"○ 사업기간 : '25. 3. ~ 12.\n\n○ 지원대상 : 울산 거주 미취업 청년\n\n○ 지원인원 : 50명\n\n○ 지원내용 : 공공기관 취업을 위한 맞춤형 패키지 교육\n\n○ 수행기관 : 울산경제일자리진흥원","sprvsnInstCd":"6310000","sprvsnInstCdNm":"울산광역시","sprvsnInstPicNm":"경제정책관 일자리지원팀","operInstCd":"B554992","operInstCdNm":"재단법인울산경제일자리진흥원","operInstPicNm":"울산경제일자리진흥원","sprtSclLmtYn":"N","aplyPrdSeCd":"0057002","bizPrdSeCd":"0056001","bizPrdBgngYmd":"20250401","bizPrdEndYmd":"20251231","bizPrdEtcCn":"","plcyAplyMthdCn":"","srngMthdCn":"","aplyUrlAddr":"

In [ ]:
import requests
import json
import pandas as pd
from tqdm import tqdm  # 진행률 표시

# 본인의 API 키 입력
API_KEY = "1bc6d1da-5d01-4bd7-8e95-805197693ab0"
BASE_URL = "https://www.youthcenter.go.kr/go/ythip/getPlcy"

# 1. 전체 정책 수 확인
init_params = {
    'apiKeyNm': API_KEY,
    'pageNum': 1,
    'pageSize': 1,  # 1개만 불러서 전체 수 확인
    'rtnType': 'json'
}
response = requests.get(BASE_URL, params=init_params)
data = response.json()
total_count = data['result']['pagging']['totCount']
print(f"✅ 총 정책 수: {total_count}")

# 2. 페이지 수 계산
page_size = 100  # 최대 효율
total_pages = (total_count + page_size - 1) // page_size
print(f"📄 총 페이지 수: {total_pages}")

# 3. 모든 페이지 반복 호출
all_policies = []
for page in tqdm(range(1, total_pages + 1), desc="정책 수집 중"):
    params = {
        'apiKeyNm': API_KEY,
        'pageNum': page,
        'pageSize': page_size,
        'rtnType': 'json'
    }

    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        data = response.json()
        policies = data.get("result", {}).get("youthPolicyList", [])
        all_policies.extend(policies)
    else:
        print(f"❌ {page}페이지 요청 실패 (status {response.status_code})")

print(f"✅ 총 수집된 정책 수: {len(all_policies)}")

# 4. 저장 (JSON)
with open("youth_policies_all1.json", "w", encoding="utf-8") as f:
    json.dump(all_policies, f, ensure_ascii=False, indent=2)

# 5. 저장 (CSV)
df = pd.DataFrame(all_policies)

# 주요 열 필터링 (존재하는 것만)
selected_cols = [
    'plcyNm', 'plcyExplnCn', 'plcySprtCn', 'aplyYmd',
    'aplyUrlAddr', 'sprtTrgtMinAge', 'sprtTrgtMaxAge', 
    'lclsfNm', 'mclsfNm'
]
existing_cols = [col for col in selected_cols if col in df.columns]
df[existing_cols].to_csv("youth_policies_all1.csv", index=False, encoding="utf-8-sig")

print("📁 CSV 및 JSON 저장 완료! ✅")


✅ 총 정책 수: 3678
📄 총 페이지 수: 37


정책 수집 중: 100%|██████████| 37/37 [00:18<00:00,  2.05it/s]


✅ 총 수집된 정책 수: 3678
📁 CSV 및 JSON 저장 완료! ✅


In [19]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("youth_policies_all.csv", encoding="utf-8-sig")

# 특정 컬럼의 고유값 보기 (예: 'lclsfNm' 정책 대분류명)
print("정책 대분류명 고유값 목록:")
print(df['lclsfNm'].dropna().unique())
print("중분류:", df['mclsfNm'].dropna().unique())

정책 대분류명 고유값 목록:
['일자리' '교육' '주거' '복지문화' '교육,교육' '일자리,교육' '참여권리' '참여권리,참여권리' '주거,주거'
 '일자리,일자리' '복지문화,복지문화' '주거,복지문화' '일자리,교육,복지문화' '일자리,복지문화' '일자리,교육,일자리'
 '참여권리,참여권리,복지문화' '교육,복지문화,교육' '교육,일자리,교육' '교육,교육,교육' '일자리,교육,교육'
 '교육,교육,복지문화' '복지문화,참여권리' '참여권리,참여권리,참여권리' '일자리,참여권리' '교육,복지문화'
 '참여권리,복지문화' '주거,참여권리' '일자리,참여권리,참여권리' '교육,참여권리,교육' '교육,주거,교육' '주거,일자리'
 '일자리,교육,참여권리' '참여권리,일자리' '일자리,주거' '일자리,교육,참여권리,참여권리' '복지문화,일자리' '교육,일자리'
 '복지문화,주거' '복지문화,교육' '교육,참여권리' '일자리,주거,교육' '교육,주거' '주거,교육']
중분류: ['취업' '창업' '미래역량강화' '전월세 및 주거급여 지원' '취약계층 및 금융지원' '교육비지원' '미래역량강화,온라인교육'
 '재직자' '건강' '취업,미래역량강화' '온라인교육' '청년참여' '청년국제교류' '주택 및 거주지' '문화활동'
 '정책인프라구축' '청년참여,정책인프라구축' '예술인지원' '권익보호' '기숙사' '주택 및 거주지,전월세 및 주거급여 지원'
 '취업,재직자' '예술인지원,문화활동' '주택 및 거주지,예술인지원' '취업,미래역량강화,예술인지원' '청년참여,청년국제교류'
 '건강,예술인지원' '재직자,문화활동' '재직자,창업' '정책인프라구축,청년국제교류' '취업,미래역량강화,재직자'
 '청년참여,정책인프라구축,예술인지원' '취약계층 및 금융지원,예술인지원' '미래역량강화,건강,온라인교육'
 '미래역량강화,창업,온라인교육' '미래역량강화,교육비지원,온라인교육' '전월세 및 주거급여 지원,예술인지원' '재직자,예술인지원'
 '취업,미래역량강화,온라인교육' '미래

In [6]:
import pandas as pd

# CSV 파일 불러오기
df = pd.read_csv("youth_policies_all.csv", encoding="utf-8-sig")

# '귀농' 또는 '귀촌'이 설명에 포함된 항목 필터링 (대소문자 무시)
filtered = df[df['plcyExplnCn'].str.contains('청년농업', case=False, na=False)]

# 상위 30개 출력
print(filtered[['plcyNm', 'plcyExplnCn']].head(30))


                               plcyNm  \
291                 춘천시 청년농업인 영농정착 지원   
294                  4-H 청년농업인 기초영농지원   
295                       청년농업인 육성지원    
298                   청년농업인 영농창업기반 조성   
300                   청년농업인 맞춤형 사업 지원   
301                청년농업인 경영실습 임대농장 조성   
373                     청년농업인 영농정착 지원   
468                  청년농업인 스마트팜 기반 조성   
557                청년농업인 스마트팜 자립기반 구축   
559            청년농업인 창업 스케일업(성장) 지원사업   
560             청년농업인 스타트업(초기창업) 지원사업   
561                     청년후계농 영농정착 지원   
667                청년농업인 특화작목 성공모델 육성   
668              청년농업인 드론활용 농작업지원단 운영   
669                신기술접목 차세대 영농인 육성지원   
730            청년농업인 맞춤형 창업 성공모델 지원사업   
731                영농승계 청년농업인 육성 지원사업   
732                  영세창업농 초기영농비 지원사업   
788                      청년농업인 영농정착지원   
790                  청년농업인 육성 종합체계 구축   
791                      청년창업농 디딤돌 사업   
792           청년농업인 후원결연 멘토링 지범 시범 사업   
1051  2024년 전남 해남군 영농승계 청년농업인 육성 지원사업   
1708       2024년